In [5]:
#load dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

path = "../data/raw/*.csv"

files = glob.glob(path)

files

[]

In [6]:
#merge 12 pond datasets into 1
import os

dataframes = []

for file in files:
    
    df = pd.read_csv(file)

    filename = os.path.basename(file)

    pond_number = ''.join(filter(str.isdigit, filename))

    df["pond"] = pond_number

    dataframes.append(df)

data = pd.concat(dataframes, ignore_index=True)

data.head()

ValueError: No objects to concatenate

In [ ]:
data.shape

(1114970, 43)

In [ ]:
#saved in process folder
data.to_csv("../data/processed/combined_pond_dataset.csv", index=False)

NameError: name 'data' is not defined

PREPROCESS DATA

In [ ]:
data.columns

NameError: name 'data' is not defined

DATA CLEANING

In [ ]:
data.columns = data.columns.str.lower()
data.columns
[c for c in data.columns if "temp" in c]
[c for c in data.columns if "turb" in c]
[c for c in data.columns if "oxygen" in c]
[c for c in data.columns if "ammo" in c]
[c for c in data.columns if "nitra" in c]

['nitrate(g/ml)', 'nitrate', 'nitrate(g/ml)', 'nitrate (mg/l)']

In [ ]:
data["temperature"] = data.filter(like="temp").iloc[:,0]
data["turbidity"] = data.filter(like="turb").iloc[:,0]
data["dissolved_oxygen"] = data.filter(like="oxygen").iloc[:,0]
data["ph"] = data.filter(like="ph").iloc[:,0]
data["ammonia"] = data.filter(like="ammo").iloc[:,0]
data["nitrate"] = data.filter(like="nitra").iloc[:,0]

In [ ]:
data = data[[
    "created_at",
    "temperature",
    "turbidity",
    "dissolved_oxygen",
    "ph",
    "ammonia",
    "nitrate",
    "pond"
]]
data.head()

,created_at,temperature,turbidity,dissolved_oxygen,ph,ph,ammonia,nitrate,pond
0,2021-06-19 00:00:05 CET,24.8750,100.0,4.505,8.43365,8.43365,0.45842,193.0,1
1,2021-06-19 00:01:02 CET,24.9375,100.0,6.601,8.43818,8.43818,0.45842,194.0,1
2,2021-06-19 00:01:22 CET,24.8750,100.0,15.797,8.42457,8.42457,0.45842,192.0,1
3,2021-06-19 00:01:44 CET,24.9375,100.0,5.046,8.43365,8.43365,0.45842,193.0,1
4,2021-06-19 00:02:07 CET,24.9375,100.0,38.407,8.40641,8.40641,0.45842,192.0,1


EDA 2.0

In [ ]:
data.head()
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1114970 entries, 0 to 1114969
Data columns (total 9 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   created_at        1044225 non-null  object 
 1   temperature       326119 non-null   float64
 2   turbidity         655652 non-null   float64
 3   dissolved_oxygen  757157 non-null   float64
 4   ph                1107513 non-null  float64
 5   ph                1107513 non-null  float64
 6   ammonia           756853 non-null   float64
 7   nitrate           757157 non-null   float64
 8   pond              1114970 non-null  object 
dtypes: float64(7), object(2)
memory usage: 76.6+ MB


In [ ]:
#check for missing values
data.isnull().sum()
#if missing, interpolate and type this command data = data.interpolate(numeric_only=True)
# if >0 = missing values

created_at           70745
temperature         788851
turbidity           459318
dissolved_oxygen    357813
ph                    7457
ph                    7457
ammonia             358117
nitrate             357813
pond                     0
dtype: int64

In [ ]:
#interpolate for missing values 1.1
data.dtypes

created_at           object
temperature         float64
turbidity           float64
dissolved_oxygen    float64
ph                  float64
ph                  float64
ammonia             float64
nitrate             float64
pond                 object
dtype: object

In [ ]:
#interpolate for missing values 1.2
data = data.loc[:, ~data.columns.duplicated()]

In [ ]:
data["created_at"] = pd.to_datetime(data["created_at"], errors="coerce", utc=True)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_19380\1979743551.py:1: FutureWarning: Parsed string "2021-06-19 00:00:05 CET" included an un-recognized timezone "CET". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  data["created_at"] = pd.to_datetime(data["created_at"], errors="coerce", utc=True)


In [ ]:
data.dtypes

created_at          datetime64[ns, UTC]
temperature                     float64
turbidity                       float64
dissolved_oxygen                float64
ph                              float64
ammonia                         float64
nitrate                         float64
pond                             object
dtype: object

EDA

In [ ]:
#SORT TIME SERIES
data = data.sort_values("created_at")

In [ ]:

#CHECK AGAIN FOR MISSING VALUES (PAGOD NA KO HAYS...)
data.isnull().sum()

created_at          288340
temperature         788851
turbidity           459318
dissolved_oxygen    357813
ph                    7457
ammonia             358117
nitrate             357813
pond                     0
dtype: int64

In [ ]:
#interpolate missi
numeric_cols = [
    "temperature",
    "turbidity",
    "dissolved_oxygen",
    "ph",
    "ammonia",
    "nitrate"
]

data[numeric_cols] = data[numeric_cols].interpolate()

In [ ]:
data = data.dropna(subset=["created_at"])

In [ ]:
#interpolation successful, no more missing values
data.isnull().sum()

created_at             0
temperature         2921
turbidity              0
dissolved_oxygen       0
ph                     0
ammonia                0
nitrate                0
pond                   0
dtype: int64

In [ ]:
#current dataset size
data.shape

(826630, 8)

In [ ]:
data.to_csv("../data/processed/combined_pond_dataset.csv", index=False)